# Preprocessing & First Model

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import min, max, xxhash64, col, when
from functools import reduce
import glob
import matplotlib.pyplot as plt
import pandas as pd
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from IPython.display import display
from pyspark.ml.feature import Imputer
from pyspark.ml.classification import RandomForestClassifier, RandomForestClassificationModel
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.mllib.evaluation import MulticlassMetrics
from pyspark.storagelevel import StorageLevel
from pyspark.ml.classification import GBTClassifier

import os
job_scratch = os.environ["TMPDIR"]  # /scratch/lcheng7/job_#
spark_tmp = os.path.join(job_scratch, "spark_tmp")
os.makedirs(spark_tmp, exist_ok=True)

# Change on creation
TOTAL_CORES = 8
TOTAL_MEM_GB = 128
DRIVER_MEM_GB = 12 # Fixed

EXEC_INSTANCES = TOTAL_CORES - 1
EXEC_MEM_GB = int((TOTAL_MEM_GB - DRIVER_MEM_GB) / EXEC_INSTANCES)

spark = (
    SparkSession.builder
    .appName("fhvhv-model")
    .config("spark.driver.memory", f"{DRIVER_MEM_GB}g")
    .config("spark.local.dir", spark_tmp)
    .config("spark.executor.instances", str(EXEC_INSTANCES))
    .config("spark.executor.memory", f"{EXEC_MEM_GB}g")
    .config("spark.sql.shuffle.partitions", str(EXEC_INSTANCES * 3))
    .config("spark.default.parallelism", str(EXEC_INSTANCES * 3))
    .getOrCreate()
)

print("TMPDIR =", job_scratch)
print("spark.local.dir =", spark.sparkContext.getConf().get("spark.local.dir"))
print("parallelism =", spark.sparkContext.defaultParallelism)
print("executors:", EXEC_INSTANCES, "executor_mem_gb:", EXEC_MEM_GB)

Matplotlib created a temporary cache directory at /scratch/spanchapakesan/job_47406209/matplotlib-6ni7__ug because the default path (/home/jovyan/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


TMPDIR = /scratch/spanchapakesan/job_47406209
spark.local.dir = /scratch/spanchapakesan/job_47406209/spark_tmp
parallelism = 21
executors: 7 executor_mem_gb: 16


In [2]:
print("MASTER =", spark.sparkContext.master)
print("DRIVER MEM =", spark.sparkContext.getConf().get("spark.driver.memory"))
print("EXEC INSTANCES =", spark.sparkContext.getConf().get("spark.executor.instances"))

MASTER = local[*]
DRIVER MEM = 12g
EXEC INSTANCES = 7


In [4]:
df_clean = spark.read.parquet("cleaned_dataset/fhvhv_dedup")
df = df_clean

df = df.drop(
    "originating_base_num",
    "on_scene_datetime",
    "dispatching_base_num",
    "request_datetime"
)

In [5]:
df_small = df.limit(5).toPandas()
df_small

,hvfhs_license_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,trip_time,base_passenger_fare,tolls,bcf,sales_tax,congestion_surcharge,airport_fee,tips,driver_pay,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag
0,HV0003,2022-09-16 13:02:34,2022-09-16 14:18:10,3,1,28.72,4536,101.50,28.58,3.98,0.0,0.0,2.5,0.00,78.13,N,N,,N,N
1,HV0003,2021-07-11 12:24:13,2021-07-11 13:06:10,4,1,13.70,2517,48.38,21.00,2.16,0.0,0.0,2.5,11.08,39.95,N,N,,N,N
2,HV0003,2022-03-13 10:09:12,2022-03-13 10:42:26,4,1,14.34,1994,50.64,21.00,2.22,0.0,0.0,2.5,0.00,39.38,N,N,,N,N
3,HV0003,2019-06-27 14:37:12,2019-06-27 15:46:35,4,1,14.41,4163,74.80,21.00,0.00,0.0,0.0,NaN,0.00,77.62,N,N,,N,
4,HV0003,2022-06-12 12:56:31,2022-06-12 13:34:47,4,1,14.42,2296,58.49,21.00,2.46,0.0,0.0,2.5,0.00,42.73,N,N,,N,N


## Imputing

Missing values were concentrated in policy-related fields such as airport_fee and congestion_surcharge.

For this dataset, a missing airport fee logically means the trip was not an airport trip, so it is reasonable to treat missing values as zero.

Similarly, missing congestion surcharge values likely indicate that the surcharge did not apply to that trip.

Rather than dropping millions of rows, we impute these values to preserve data volume and avoid introducing bias.

In [6]:
df = df.fillna({
    "airport_fee": 0,
    "congestion_surcharge": 0
})

df = df.withColumn(
    "wav_match_flag",
    F.when(
        F.col("wav_match_flag").isNull() |
        (F.length(F.trim(F.col("wav_match_flag").cast("string"))) == 0),
        F.lit("Unknown")
    ).otherwise(F.trim(F.col("wav_match_flag").cast("string")))
)

df.groupBy("wav_match_flag").count().show()

+--------------+---------+
|wav_match_flag|    count|
+--------------+---------+
|             Y| 26977884|
|             N|607638261|
|       Unknown|110657176|
+--------------+---------+



In [7]:
null_summary = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
])

null_summary.toPandas().T

,0
hvfhs_license_num,0
pickup_datetime,0
dropoff_datetime,0
PULocationID,0
DOLocationID,0
trip_miles,0
trip_time,0
base_passenger_fare,0
tolls,0
bcf,0


## Scaling/Transforming

In [8]:
num_cols = [
    "log_fare",
    "log_miles",
    "log_minutes",
    "tolls",
    "congestion_surcharge",
    "airport_fee",
    "pickup_hour",
    "pickup_dow",
    "pickup_month",
    "is_weekend",
    "is_night",
    "has_airport_fee",
    "has_congestion_fee",
    "total_surcharges",
    "fare_per_mile"
]

# target (tip)
df1 = df.withColumn("tip_binary", F.when(F.col("tips").cast("double") > 0, 1).otherwise(0))

# numeric sanity casts first
df1 = (
    df1
    .withColumn("base_passenger_fare", F.col("base_passenger_fare").cast("double"))
    .withColumn("trip_miles", F.col("trip_miles").cast("double"))
    .withColumn("trip_time", F.col("trip_time").cast("double"))
    .withColumn("tolls", F.col("tolls").cast("double"))
    .withColumn("bcf", F.col("bcf").cast("double"))
    .withColumn("sales_tax", F.col("sales_tax").cast("double"))
    .withColumn("congestion_surcharge", F.col("congestion_surcharge").cast("double"))
    .withColumn("airport_fee", F.col("airport_fee").cast("double"))
    .withColumn("tips", F.col("tips").cast("double"))
)

# time features
df1 = (
    df1
    .withColumn("trip_minutes", F.col("trip_time") / F.lit(60.0))
    .withColumn("pickup_hour", F.hour("pickup_datetime").cast("double"))
    .withColumn("pickup_dow", F.dayofweek("pickup_datetime").cast("double"))
    .withColumn("pickup_month", F.month("pickup_datetime").cast("double"))
    .withColumn("log_fare", F.when(F.col("base_passenger_fare") >= 0, F.log1p("base_passenger_fare")).otherwise(None))
    .withColumn("log_miles", F.when(F.col("trip_miles") >= 0, F.log1p("trip_miles")).otherwise(None))
    .withColumn("log_minutes", F.when(F.col("trip_minutes") >= 0, F.log1p("trip_minutes")).otherwise(None))
)

# flags
df1 = (
    df1
    .withColumn("is_weekend", F.when(F.dayofweek("pickup_datetime").isin([1, 7]), 1.0).otherwise(0.0))
    .withColumn("is_night", F.when(F.hour("pickup_datetime").isin(list(range(0,6)) + list(range(20,24))), 1.0).otherwise(0.0))
    .withColumn("has_airport_fee", F.when(F.col("airport_fee") > 0, 1.0).otherwise(0.0))
    .withColumn("has_congestion_fee", F.when(F.col("congestion_surcharge") > 0, 1.0).otherwise(0.0))
)

df1 = (
    df1
    .withColumn(
        "total_surcharges",
        F.coalesce("tolls", F.lit(0.0)) +
        F.coalesce("bcf", F.lit(0.0)) +
        F.coalesce("sales_tax", F.lit(0.0)) +
        F.coalesce("congestion_surcharge", F.lit(0.0)) +
        F.coalesce("airport_fee", F.lit(0.0))
    )
)

df1 = (
    df1
    .withColumn(
        "fare_per_mile",
        F.when(F.col("trip_miles") > 0, F.col("base_passenger_fare") / F.col("trip_miles")).otherwise(None)
    )
)

df1.select(
    "tip_binary", "log_fare", "log_miles", "log_minutes", "pickup_hour", 
    "pickup_dow", "pickup_month","is_weekend","is_night",
    "has_airport_fee","has_congestion_fee",
    "total_surcharges", "fare_per_mile"
).limit(5).toPandas()

,tip_binary,log_fare,log_miles,log_minutes,pickup_hour,pickup_dow,pickup_month,is_weekend,is_night,has_airport_fee,has_congestion_fee,total_surcharges,fare_per_mile
0,0,4.629863,3.391820,4.338597,13.0,6.0,9.0,0.0,0.0,1.0,0.0,35.06,3.534123
1,1,3.899545,2.687847,3.760037,12.0,1.0,7.0,1.0,0.0,1.0,0.0,25.66,3.531387
2,0,3.944297,2.730464,3.533200,10.0,1.0,3.0,1.0,0.0,1.0,0.0,25.72,3.531381
3,0,4.328098,2.735017,4.253956,14.0,5.0,6.0,0.0,0.0,0.0,0.0,21.00,5.190840
4,0,4.085808,2.735665,3.670376,12.0,1.0,6.0,1.0,0.0,1.0,0.0,25.96,4.056172


In [9]:
# 1. Convert NaNs to Nulls without changing column names
df_num = df1.replace(float('nan'), None, subset=num_cols)

# 2. Imputation
imputer = Imputer(
    inputCols=num_cols,
    outputCols=[f"{c}_imp" for c in num_cols],
    strategy="median"
)
imputer_model = imputer.fit(df_num)
df_imp = imputer_model.transform(df_num)

# 3. Assemble
assembler_num = VectorAssembler(
    inputCols=[f"{c}_imp" for c in num_cols],
    outputCol="num_features",
    handleInvalid="error"
)
df2 = assembler_num.transform(df_imp)

# 4. Scale: Standardize the features
scaler = StandardScaler(
    inputCol="num_features", 
    outputCol="num_scaled", 
    withMean=True, 
    withStd=True
)
scaler_model = scaler.fit(df2)
df3 = scaler_model.transform(df2)

# 5. View
df3.select("num_features", "num_scaled", "tip_binary").limit(5).toPandas()

,num_features,num_scaled,tip_binary
0,"[4.629862798578463, 3.391820219849558, 4.33859...","[2.7134588173097325, 2.775032795508785, 2.5435...",0
1,"[3.8995454839170334, 2.6878474937846906, 3.760...","[1.6223390398921094, 1.7474890769302869, 1.593...",1
2,"[3.944296565978442, 2.73046379593911, 3.533199...","[1.6891987318613169, 1.8096933527018768, 1.221...",0
3,"[4.328098292648326, 2.7350166493320245, 4.2539...","[2.2626120224536175, 1.8163388600237997, 2.404...",0
4,"[4.085808231199814, 2.7356653681351832, 3.6703...","[1.9006221413280942, 1.81728575314427, 1.44631...",0


In [10]:
print("Rows:", df3.count())
df3.groupBy("tip_binary").count().show()

Rows: 745273321
+----------+---------+
|tip_binary|    count|
+----------+---------+
|         0|628436051|
|         1|116837270|
+----------+---------+



## Encoding

In [11]:
# Categorical

cat_cols = [
    "hvfhs_license_num",
    "shared_request_flag",
    "shared_match_flag",
    "access_a_ride_flag",
    "wav_request_flag",
    "wav_match_flag"
]

df5 = df3

for c in cat_cols:
    df5 = df5.withColumn(c, F.trim(F.col(c).cast("string")))
    df5 = df5.withColumn(
        c,
        F.when(F.col(c).isNull() | (F.length(F.col(c)) == 0), F.lit("Unknown")).otherwise(F.col(c))
    )

for c in cat_cols:
    print(c)
    pdf = df5.groupBy(c).count().orderBy(F.desc("count")).limit(10).toPandas()
    display(pdf)

hvfhs_license_num


,hvfhs_license_num,count
0,HV0003,533991717
1,HV0005,191007781
2,HV0004,13884889
3,HV0002,6388934


shared_request_flag


,shared_request_flag,count
0,N,683038777
1,Y,62234543
2,Unknown,1


shared_match_flag


...

,access_a_ride_flag,count
0,Unknown,479489134
1,N,265784187


wav_request_flag


,wav_request_flag,count
0,N,744488730
1,Y,784590
2,Unknown,1


wav_match_flag


,wav_match_flag,count
0,N,607638261
1,Unknown,110657176
2,Y,26977884


In [12]:
# StringIndex + OneHot Encode

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in cat_cols
]

encoder = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in cat_cols],
    outputCols=[f"{c}_ohe" for c in cat_cols],
    handleInvalid="keep"
)

df_idx = df5
for idx in indexers:
    df_idx = idx.fit(df_idx).transform(df_idx)

df_enc = encoder.fit(df_idx).transform(df_idx)

In [13]:
# Check new dataset
df_enc.select(
    cat_cols +
    [f"{c}_idx" for c in cat_cols] +
    [f"{c}_ohe" for c in cat_cols]
).limit(3).toPandas().T

,0,1,2
hvfhs_license_num,HV0003,HV0003,HV0003
shared_request_flag,N,N,N
shared_match_flag,N,N,N
access_a_ride_flag,Unknown,Unknown,Unknown
wav_request_flag,N,N,N
wav_match_flag,N,N,N
hvfhs_license_num_idx,0.0,0.0,0.0
shared_request_flag_idx,0.0,0.0,0.0
shared_match_flag_idx,0.0,0.0,0.0
access_a_ride_flag_idx,0.0,0.0,0.0


## Feature Engineering

In [14]:
assembler_all = VectorAssembler(
    inputCols=["num_scaled"] + [f"{c}_ohe" for c in cat_cols],
    outputCol="features",
    handleInvalid="keep"
)

df_final = assembler_all.transform(df_enc).select(
    F.col("tip_binary").cast("int").alias("label"),
    "features",
    "pickup_datetime",
    "PULocationID"
)

df_final.limit(5).toPandas()

,label,features,pickup_datetime,PULocationID
0,0,"(2.7134588173097325, 2.775032795508785, 2.5435...",2022-09-16 13:02:34,3
1,1,"(1.6223390398921094, 1.7474890769302869, 1.593...",2021-07-11 12:24:13,4
2,0,"(1.6891987318613169, 1.8096933527018768, 1.221...",2022-03-13 10:09:12,4
3,0,"(2.2626120224536175, 1.8163388600237997, 2.404...",2019-06-27 14:37:12,4
4,0,"(1.9006221413280942, 1.81728575314427, 1.44631...",2022-06-12 12:56:31,4


In [ ]:
df_final.write.mode("overwrite").parquet("processed_data/vectorized_features")

# Modeling

In [3]:
df_final = spark.read.parquet("processed_data/vectorized_features")

In [4]:
df_model = df_final.select("label", "features")

In [5]:
# Verify class imbalance
df_model = df_final.select("label", "features")
df_model.groupBy("label").count().show()

+-----+---------+
|label|    count|
+-----+---------+
|    0|628436051|
|    1|116837270|
+-----+---------+



In [6]:
# Stratified 30% subsample
df_model = df_model.sampleBy("label", fractions={0: 0.3, 1: 0.3}, seed=42)

In [7]:
# Verify class imbalance of subsample
df_model.groupBy("label").count().show()

+-----+---------+
|label|    count|
+-----+---------+
|    0|188519613|
|    1| 35053289|
+-----+---------+



In [8]:
# Calculate balancing ratio
total = 628436051 + 116837270
pos   = 116837270
balancing_ratio = (total - pos) / pos  # ~5.38

# Add class weight column
df_model = df_model.withColumn(
    "class_weight",
    when(col("label") == 1, balancing_ratio).otherwise(1.0)
)

# Split
train_df, test_df = df_model.randomSplit([0.8, 0.2], seed=42)

In [9]:
print("Train size:", train_df.count())
print("Test size:", test_df.count())
train_df.groupBy("label").count().show()
test_df.groupBy("label").count().show()

Train size: 178851498
Test size: 44721404
+-----+---------+
|label|    count|
+-----+---------+
|    0|150806846|
|    1| 28044652|
+-----+---------+

+-----+--------+
|label|   count|
+-----+--------+
|    0|37712767|
|    1| 7008637|
+-----+--------+



In [ ]:
train_df.write.mode("overwrite").parquet("train_split")
test_df.write.mode("overwrite").parquet("test_split")

In [5]:
train_df = spark.read.parquet("train_split")
test_df = spark.read.parquet("test_split")

## SVD (Singular Value Decomposition)

In [4]:
from pyspark.mllib.linalg import Vectors as MLLibVectors
from pyspark.mllib.linalg.distributed import RowMatrix

# Convert train_df "features" (pyspark.ml.linalg) to RDD of mllib Vectors for RowMatrix
def ml_to_mllib(v):
    return MLLibVectors.dense(v.toArray())

features_rdd = train_df.select("features").rdd.map(lambda row: ml_to_mllib(row.features))
row_matrix = RowMatrix(features_rdd)

num_features = len(train_df.first().features)
# keeping k as min of features or 20
k = 20 if num_features >= 20 else num_features  

svd = row_matrix.computeSVD(k, computeU=True)

# Results: U (left singular vectors), s (singular values), V (right singular vectors)
print("Number of singular values (k):", len(svd.s))
print("Singular values (first 10):", svd.s[:10] if hasattr(svd.s, '__getitem__') else svd.s.toArray()[:10])
print("V shape (right singular vectors):", svd.V.numRows, "x", svd.V.numCols)


Number of singular values (k): 20
Singular values (first 10): [28901.33477397 26745.90589668 18974.3760086  16838.44472647
 14157.41108726 13831.47166967 13676.576221   13300.71944635
 12962.81255853 12333.12695947]
V shape (right singular vectors): 39 x 20


In [5]:
# Save SVD to disk (V and s; U is distributed and not needed for projecting new data)
from pyspark.sql.types import StructType, StructField, IntegerType, ArrayType, DoubleType

# Save V (right singular vectors) - needed to project any row onto the SVD space
v_rows, v_cols = svd.V.numRows, svd.V.numCols
v_values = [float(x) for x in svd.V.values]  # Python floats (Spark can't infer numpy float64)
schema_v = StructType([
    StructField("numRows", IntegerType()),
    StructField("numCols", IntegerType()),
    StructField("values", ArrayType(DoubleType()))
])
spark.createDataFrame([(v_rows, v_cols, v_values)], schema_v).write.mode("overwrite").parquet("processed_data/svd_V")

# Save s (singular values)
s_arr = svd.s.toArray().tolist() if hasattr(svd.s, "toArray") else list(svd.s)
s_values = [float(x) for x in s_arr]  # Python floats
schema_s = StructType([StructField("singular_values", ArrayType(DoubleType()))])
spark.createDataFrame([(s_values,)], schema_s).write.mode("overwrite").parquet("processed_data/svd_s")

PySparkTypeError: [CANNOT_INFER_TYPE_FOR_FIELD] Unable to infer the type of the field `values`.

In [3]:
from pyspark.mllib.linalg import DenseMatrix, DenseVector
df_v = spark.read.parquet("processed_data/svd_V")
r = df_v.first()
svd_V = DenseMatrix(r.numRows, r.numCols, r.values)
df_s = spark.read.parquet("processed_data/svd_s")
svd_s = DenseVector(df_s.first().singular_values)


In [ ]:
# Additional analysis: project data onto top-k SVD components (SVD loaded from disk)

from pyspark.ml.linalg import Vectors as MLVectors
from pyspark.sql import Row
from pyspark.mllib.linalg import Vectors as MLLibVectors
from pyspark.mllib.linalg.distributed import RowMatrix
from pyspark.mllib.linalg import DenseMatrix

# Load SVD V from disk
df_v = spark.read.parquet("processed_data/svd_V")
r = df_v.first()
V = DenseMatrix(r.numRows, r.numCols, r.values)
num_features, k = V.numRows, V.numCols

# Build row_matrix from train_df
def ml_to_mllib(v):
    return MLLibVectors.dense(v.toArray())
features_rdd = train_df.select("features").rdd.map(lambda row: ml_to_mllib(row.features))
row_matrix = RowMatrix(features_rdd)

# Project: each row (1 x n) * V (n x k) -> (1 x k) reduced feature vector
reduced_row_matrix = row_matrix.multiply(V)

# Align reduced features with labels using row index 
reduced_rdd = reduced_row_matrix.rows.zipWithIndex()
labels_rdd = train_df.select("label").rdd.zipWithIndex()

reduced_rdd_keyed = reduced_rdd.map(lambda x: (x[1], x[0]))
labels_rdd_keyed = labels_rdd.map(lambda x: (x[1], x[0]))
joined = reduced_rdd_keyed.join(labels_rdd_keyed).map(lambda x: (x[1][0].toArray().tolist(), x[1][1]))

train_reduced_df = spark.createDataFrame(
    joined.map(lambda x: Row(features=MLVectors.dense(x[0]), label=x[1])),
    ["features", "label"]
)

print("Original feature dim:", num_features, "-> Reduced (SVD) dim:", k)
train_reduced_df.select("features", "label").limit(3).show(truncate=False)

